这个notebook专注于理解下面这行代码。对，就这一行。

In [3]:
from turtle import Turtle, Screen

## 1 **`import Turtle`**
从turtle library里导入Turtle class。

对着`Turtle`点右键，再Go To Declaration or Usages后跳出来的是这个：

```python
class Turtle(RawTurtle):
    """RawTurtle auto-creating (scrolled) canvas.

    When a Turtle object is created or a function derived from some
    Turtle method is called a TurtleScreen object is automatically created.
    """
    _pen = None
    _screen = None

    def __init__(self,
                 shape=_CFG["shape"],
                 undobuffersize=_CFG["undobuffersize"],
                 visible=_CFG["visible"]):
        if Turtle._screen is None:
            Turtle._screen = Screen()
        RawTurtle.__init__(self, Turtle._screen,
                           shape=shape,
                           undobuffersize=undobuffersize,
                           visible=visible)
```

## 2 **`import Screen`**
对着line1的`Screen`点右键，再Go To Declaration or Usages后跳出来的是这个：

```python
def Screen():
    """
    Return the singleton screen object.                              翻译：返回单例screen对象
    If none exists at the moment, create a new one and return it,    翻译：如果当下窗口对象还不存在，就新创建一个窗口对象，并将其返回
    else return the existing one.                                    翻译：否则就返回现有的窗口对象
    """
    if Turtle._screen is None:
        Turtle._screen = _Screen()
    return Turtle._screen
```

而且下面还紧跟着一坨这个：

```python
class _Screen(TurtleScreen):
    ...
```


Screen()是`turtle library`中的一个**函数**，它不隶属于任何一个`class`。

这个函数内部的注释写得很有意思，只是语焉不详，不利于理解。

|注释里的词|实际指的是|
|---|---|
|none exists|`Turtle._screen is None`，即窗口对象还不存在|
|a new one|新创建的 `_Screen()` 实例（窗口对象）|
|it|同上，那个新建的窗口对象|
|the existing one|已经存在于 `Turtle._screen` 里的那个窗口对象|


值得关注的是`_screen`这个变量。

### 2.1 这个见鬼的 **`_screen`** 是啥？答：这是一个**类对象**。

#### 2.1.1 实例对象

比如下面的代码里的`color`就是一个实例变量。它**既写在class里，又存在于某个方法里**。

```python
class Turtle:
    def __init__(self):
        self.color = "black"   # 每只乌龟自己的颜色
```

`tim = Turtle()` → `tim.color = "black"` 

`bob = Turtle()` → `bob.color = "black"`

这俩 `color` **互不影响**，各自独立。

#### 2.1.2 类对象
类变量就是**写在 class 里、但不在任何方法里**的变量：

```python
class Turtle:
    _screen = None    # 类变量，在所有方法之外
    
    def __init__(self):
        self.color = "black"   # 实例变量
```

`_screen` 不属于某只乌龟，它属于 **`Turtle` 这个 class 本身**。所有乌龟共享同一个 `_screen`。

调用时，写`Turtle._screen`。这个是直接通过**类名Turtle**调用类的变量`_screen`。

### 2.2 知道了Screen()中反复出现的_screen后，就可以读这个函数了。Singleton（单例模式）

这是一个很经典的设计模式叫 Singleton（单例模式）。

Screen 不是一个 class，而是一个**工厂函数**，但它的行为和直接实例化 class 几乎一样。原理如下：

In [6]:
def Screen():
    if Turtle._screen is None:
        Turtle._screen = _Screen()   # 真正的 class 是 _Screen（注意下划线）
    return Turtle._screen

1. 在初始状态下

`Turtle._screen`中没有东西。因为它是一个类变量，导致当我们只是调用构造方法Turtle()构建一个Turtle class的实例对象时，`Turtle._screen`不会被初始化。

这时候，去判断`Turtle._screen is None`，得到的肯定是`True`。

在这种情况下，我们去访问真正定义了“屏幕”这个概念的类：`_Screen class`的构造方法`_Screen()`，以此实例化一个`_Screen`类的对象，并将其存入`Turtle._screen`中。

然后返回这个`Turtle._screen`。

2. 在执行过一次`Screen()`后（通常是外部调用来创建一个屏幕对象）

这时候，去判断`Turtle._screen is None`，得到的是`False`。

就会直接返回在第一次调用`Screen()`时创建的`Turtle._screen`。



#### **理解程序设计思想**

实际上，在读完Screen函数内部的内容后，就会发现，它实际上就是给_Screen类的构造方法_Screen()外面套了个壳子，多处理一层逻辑。

某种程度上，它执行的是一种“过滤”的思想，保证了构造方法_Screen()只会被调用一次。之所以要这么做，是因为从逻辑上来说，就应该是：一个程序只应该有一个 turtle 画布窗口，而不能同时弹出两个窗口。

即：
- `Turtle()` → 每次调用都创建新的乌龟（可以有多只）
- `Screen()` → 永远返回同一个窗口（只有一个）

可以验证一下：

```python
s1 = Screen()
s2 = Screen()
print(s1 is s2)  # True，是同一个对象
```

返回的是True，证明 `s1` 和 `s2` 是同一个对象。

### turtle.listen(xdummy=None, ydummy=None)

> Set focus on TurtleScreen (in order to collect key-events).

1. 什么是“获取焦点(focus)”？
    当你的电脑屏幕上同时开着好几个软件（比如代码编辑器、浏览器、微信，以及 Turtle 弹出的画图窗口）时，你敲击键盘，操作系统怎么知道你要把按键指令发给哪个窗口呢？操作系统默认只会把键盘信号发给当前被激活、处于最前面的那个窗口，这个状态在编程里就叫做“拥有焦点 (Focus)”。

    screen.listen() 的真实作用： 就是命令 Turtle 的画图窗口“竖起耳朵，把焦点抢过来”。

    如果不写这一句： 你即便写了 screen.onkey(f, "Up")，当你按下向上方向键时，键盘信号可能会发给你的代码编辑器（因为你刚在里面敲完代码，焦点还在编辑器上），Turtle 窗口根本“听不到”，你的乌龟也就不会动。

> Dummy arguments are provided in order to be able to pass listen() to the onclick method.

2. 什么是“占位参数(Dummy Arguments)”？
    在 Turtle 中，如果你想用鼠标点击屏幕来触发某个动作（使用 `onclick` 方法），Turtle 会默认在后台悄悄把鼠标点击位置的 `X` 坐标和 `Y` 坐标传递给你要执行的函数。

    假设你想实现这样一个功能：“用鼠标点一下屏幕，屏幕才开始监听键盘”。你会写出类似 `screen.onclick(turtle.listen)` 的代码。

    因为 `onclick` 会硬塞两个坐标数据给 `listen`，如果 `listen` 拒收，程序就会报错崩溃。

    所以官方在设计 `listen(xdummy=None, ydummy=None)` 时，故意留了两个空位（Dummy 代表“占位符”）。它其实根本不需要这两个坐标，仅仅是为了在接收 `onclick` 传来的坐标时不报错，把坐标默默“吞掉”而已。作为初学者，日常调用时完全不需要在括号里填任何东西，直接写 `screen.listen()` 即可。


#### turtle.onkey(fun, key)
#### turtle.onkeyrelease(fun, key)

**Parameters:**
- fun – a function with no arguments or None
- key – a string: key (e.g. “a”) or key-symbol (e.g. “space”)

Bind fun to key-release event of key. If fun is None, event bindings are removed. Remark: in order to be able to register key-events, TurtleScreen must have the focus. (See method listen().)



In [ ]:
def f():                # 1. 定义了一个不需要任何参数的函数 f
    fd(50)
    lt(60)

screen.onkey(f, "Up")   # 2. 把函数 f 绑定到 "Up"（上箭头）键上
screen.listen()         # 3. 让窗口竖起耳朵准备接收按键